In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'

In [ ]:
mct = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/raw_norm.h5ad')

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc/'
m3c = sc.read_h5ad(f'{indir}/h5ad/chrom100k/raw_norm.h5ad')

In [ ]:
mct

In [ ]:
m3c

In [ ]:
m3c_meta = pd.read_csv(f'{projdir}/csv/choo_raw/B01_m3C_filtered.csv')
mct_meta = pd.read_csv(f'{projdir}/csv/choo_raw/B01_mCT_filtered.csv')
m3c_meta.shape, mct_meta.shape

In [ ]:
mct_meta.index = mct_meta['wellprefix']
m3c_meta.index = m3c_meta['wellprefix']

keep = ['time', 'line', 'DNA_mCGfrac', 'DNA_mCHfrac', 'DNA_mCCCfrac']
tmp_mct = mct_meta[keep]

keep = ['time', 'line', 'mCGfrac', 'mCHfrac', 'mCCCfrac']
tmp_m3c = m3c_meta[keep]

tmp_mct.columns = ['time', 'line','mCGfrac', 'mCHfrac', 'mCCCfrac']
tmp_mct['assay'] = 'snmCT'
tmp_m3c['assay'] = 'snm3C'

In [ ]:
mct.obs.index = [x.replace('_indexed', '') for x in mct.obs.index]
mct.obs = tmp_mct.reindex(mct.obs.index)

m3c.obs.index = [x.replace('_indexed', '') for x in m3c.obs.index]
m3c.obs = tmp_m3c.reindex(m3c.obs.index)

In [ ]:
meta = pd.concat([tmp_mct, tmp_m3c])
meta['donor_time'] = [f'{x}_{y}' for x,y in zip(meta['line'], meta['time'])]
meta.shape

In [ ]:
meta.head()

### first write out to merged

In [ ]:
adata = mct.concatenate(m3c)

In [ ]:
adata

In [ ]:
final_adata = adata[~adata.obs['line'].isna()]

In [ ]:
final_adata

In [ ]:
final_adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.raw.h5ad')

### subset to the subsampled!

In [ ]:
mct.obs['donor_time'] = [f'{x}_{y}' for x,y in zip(mct.obs['line'], mct.obs['time'])]
m3c.obs['donor_time'] = [f'{x}_{y}' for x,y in zip(m3c.obs['line'], m3c.obs['time'])]

In [ ]:
mct_dt = pd.DataFrame(mct.obs['donor_time'].value_counts().sort_index())
m3c_dt = pd.DataFrame(m3c.obs['donor_time'].value_counts().sort_index())

dt = pd.concat([mct_dt, m3c_dt], axis=1)
dt.columns = ['snmct', 'snm3c']

In [ ]:
to_drop = dt[dt['snmct'].isna()].index[:-1]
to_drop

In [ ]:
m3c_mask = [x not in to_drop for x in m3c.obs['donor_time']]
m3c = m3c[m3c_mask]

In [ ]:
# snmct_sub

In [ ]:
mct = mct[~mct.obs['time'].isna()]

In [ ]:
times = sorted(mct.obs['time'].unique())
assays = ['snmct', 'snm3c']
sub_mct_list = []
sub_m3c_list = []
for time in times:
    print(time)
    
    tmp_mct = mct[mct.obs['time'] == time]
    tmp_m3c = m3c[m3c.obs['time'] == time]

    mct_num = tmp_mct.shape[0]
    m3c_num = tmp_m3c.shape[0]
    
    print(mct_num, m3c_num)

    lower_bound = np.min([mct_num, m3c_num])
    tmp_assay = np.argmax([mct_num, m3c_num])
    
    if tmp_assay == 0:
        sc.pp.subsample(tmp_mct, n_obs=lower_bound, random_state=0)
    else:
        sc.pp.subsample(tmp_m3c, n_obs=lower_bound, random_state=0)
        
    sub_mct_list.append(tmp_mct)
    sub_m3c_list.append(tmp_m3c)

In [ ]:
final_mct = anndata.concat(sub_mct_list)
final_m3c = anndata.concat(sub_m3c_list)

In [ ]:
final_mct

In [ ]:
final_m3c

In [ ]:
final_mct.write_h5ad(f'{projdir}/h5ad/integrate_m3c/mct.raw.h5ad')
final_m3c.write_h5ad(f'{projdir}/h5ad/integrate_m3c/m3c.raw.subsample.h5ad')

In [ ]:
adata = final_mct.concatenate(final_m3c)

In [ ]:
adata = adata[~adata.obs['time'].isna()]

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.raw.subsample.h5ad')

In [ ]:
!date